In [1]:
# Setup and imports
%load_ext autoreload
%autoreload 2

import sys
import os

# Fix: Change working directory to project root for proper path resolution
original_cwd = os.getcwd()
project_root = os.path.dirname(original_cwd)  # Go up from notebooks/ to project root
os.chdir(project_root)

print(f"📂 Changed working directory from:")
print(f"   {original_cwd}")
print(f"📂 To project root:")
print(f"   {os.getcwd()}")

# Add project root to Python path (now that we're in the right directory)
sys.path.insert(0, '.')

# Import core system setup (paths will now resolve correctly)
from qanda_module.config import setup_system
from qanda_module.ui_gradio import launch_ui_with_toggle

print("📦 Imports successful!")

# System setup with correct path resolution
print("🚀 Setting up system...")

helpers, qa_chain, config = setup_system("config.yaml")  # Now works with original config paths

print("✅ System setup complete!")
print(f"Model: {config.chat_model_name}")
print(f"Temperature: {config.temperature}")  
print(f"Debug: {config.debug_enabled}")

# Optional: Change back to notebooks directory if needed for other operations
# os.chdir(original_cwd)

📂 Changed working directory from:
   /Users/swang/workspace/repos/qanda_clean1/qanda-v2/notebooks
📂 To project root:
   /Users/swang/workspace/repos/qanda_clean1/qanda-v2
📦 Imports successful!
🚀 Setting up system...
Loaded 1094 panelist profile URLs
Total panellist responses: 65031
Panellist responses with missing speaker_id: 0
Created panelist_lookup with 1094 entries for UI display
✅ System setup complete!
Model: gpt-4o-mini
Temperature: 0.3
Debug: True


In [2]:
# Debug functions for new chunk format
def debug_new_chunk_format(qa_chain, query="test"):
    """Debug what documents look like after retrieval"""
    print("🔍 DEBUGGING NEW CHUNK FORMAT")
    
    # Get raw documents
    docs = qa_chain.retriever._get_relevant_documents(query)
    
    print(f"\n📊 Retrieved {len(docs)} documents")
    
    for i, doc in enumerate(docs[:2]):  # Show first 2
        print(f"\n📄 Document {i+1}:")
        print(f"Content: {doc.page_content[:200]}...")
        print(f"Metadata keys: {list(doc.metadata.keys())}")
        print(f"Speaker type: {doc.metadata.get('speaker_type')} (type: {type(doc.metadata.get('speaker_type'))})")
        print(f"Episode ID: {doc.metadata.get('episode_id')}")
        print(f"Speaker name: {doc.metadata.get('speaker_name')}")

def test_simple_query(qa_chain, query="What did panelists say about climate change?"):
    """Test pipeline without formatting to isolate issues"""
    print("🧪 TESTING SIMPLE QUERY")
    
    docs = qa_chain.retriever._get_relevant_documents(query)
    print(f"📊 Retrieved {len(docs)} documents")
    
    context = "\n\n".join([doc.page_content for doc in docs[:5]])
    print(f"\n📄 Context preview: {context[:500]}...")
    
    # Test AI generation only
    try:
        from qanda_module.retrieval_lean import generate_ai_response
        response = generate_ai_response(docs[:5], query, "Standard", config)
        print(f"\n✅ AI Response ({len(response)} chars): {response[:200]}...")
        return True
    except Exception as e:
        print(f"\n❌ AI Generation failed: {e}")
        return False

print("🔧 Debug functions loaded!")

🔧 Debug functions loaded!


In [3]:
print("=" * 50)
debug_new_chunk_format(qa_chain, "climate change")

print("\n" + "=" * 50)

🔍 DEBUGGING NEW CHUNK FORMAT

📊 Retrieved 80 documents

📄 Document 1:
Content: [Episode ID: 212] Title: Section 44, Space and Skeptics | Topic: ANGUS TAYLOR: CLIMATE | Date: 2017-11-13 | Response from BRIAN COX: The rate of change is human driven. I think that's broadly accepted...
Metadata keys: ['episode_title', 'episode_id', 'response_id', 'speaker_type', 'question_id', 'speaker_id', 'speaker_name', 'episode_date']
Speaker type: 3 (type: <class 'int'>)
Episode ID: 212
Speaker name: BRIAN COX

📄 Document 2:
Content: [Episode ID: 461] Title: Q&A Climate Debate | Topic: PSYCHOLOGY OF CHANGING | Date: 2012-04-26 | Response from CLIVE PALMER: By 2017, you say we've got to reduce it....
Metadata keys: ['episode_title', 'speaker_name', 'response_id', 'episode_id', 'episode_date', 'question_id', 'speaker_id', 'speaker_type']
Speaker type: 3 (type: <class 'int'>)
Episode ID: 461
Speaker name: CLIVE PALMER



In [4]:
# Test 2: Test simple pipeline
success = test_simple_query(qa_chain, "What did panelists say about climate change?")

if success:
    print("\n✅ Basic pipeline works - issue is in formatting")
else:
    print("\n❌ Basic pipeline broken - issue is in retrieval/AI generation")

🧪 TESTING SIMPLE QUERY
📊 Retrieved 80 documents

📄 Context preview: [Episode ID: 344] Title: This is your Brain on Climate Change | Topic: CLIMATE PUP | Date: 2014-11-10 | Response from TONY JONES: Actually, let's hear from other panellists before we go back to the politicians. James?

[Episode ID: 555] Title: The PM on Q and A | Topic: CLIMATE/SCEPTIC | Date: 2010-02-08 | Response from KEVIN RUDD: As you can see there is a bit of division on that in the room, like the chamber up the hill. The first thing I'd say is the IPCC - International Panel on Climate Chan...

✅ AI Response (1140 chars): The panelists discussed various aspects of climate change, highlighting both scientific consensus and differing opinions. In Episode ID: 555, Kevin Rudd emphasized the findings of the IPCC, stating th...

✅ Basic pipeline works - issue is in formatting


In [5]:
# Debug the formatting stage specifically
def test_formatting_only():
    """Test just the formatting part that's failing"""
    print("🔍 TESTING FORMATTING STAGE")
    
    # Get a simple AI response
    docs = qa_chain.retriever._get_relevant_documents("climate change")[:5]
    from qanda_module.retrieval_lean import generate_ai_response
    ai_response = generate_ai_response(docs, "What did panelists say about climate change?", "Standard", config)
    
    print(f"📄 AI Response preview: {ai_response[:200]}...")
    
    try:
        print("\n🔍 Testing helpers.format_response_with_links...")
        formatted_response, sources = helpers.format_response_with_links(ai_response, docs)
        print(f"✅ Formatting succeeded!")
        print(f"📊 Formatted response length: {len(formatted_response)}")
        print(f"📊 Sources length: {len(sources)}")
        return True
        
    except Exception as e:
        import traceback
        print(f"❌ Formatting failed!")
        print(f"Error type: {type(e).__name__}")
        print(f"Error message: '{str(e)}'")
        print(f"Full traceback:\n{traceback.format_exc()}")
        return False

# Test it
test_formatting_only()

🔍 TESTING FORMATTING STAGE
📄 AI Response preview: The panelists expressed a range of views on climate change, with a consensus on its human-driven nature and urgency.

Brian Cox stated that "the rate of change is human driven" and emphasized that thi...

🔍 Testing helpers.format_response_with_links...
Speakers found in results: {'Brian Cox': 'BRIAN COX', 'Clive Palmer': 'CLIVE PALMER', 'A.C. Grayling': 'A.C. GRAYLING', 'Tanya Plibersek': 'TANYA PLIBERSEK'}
Total panelist URLs available: 1094
Found URL for BRIAN COX via Brian Cox: https://www.abc.net.au/qanda/brian-cox/11191126
  → Replaced Brian Cox using pattern: (^|[.!?]\s+|:\s*)Brian\ Cox\b...
Found URL for CLIVE PALMER via Clive Palmer: https://www.abc.net.au/qanda/clive-palmer/10642046
  → WARNING: Could not find Clive Palmer in text with any pattern
    Found at: ...21). 

In contrast, Clive Palmer's comments reflecte...
Found URL for A.C. GRAYLING via A.C. Grayling: https://www.abc.net.au/qanda/a.c.-grayling/11377530
  → Replace

False

In [6]:
# Simple analysis: check 200 chunks for missing titles
from chromadb import PersistentClient
import pandas as pd

client = PersistentClient(path=config.chroma_path)
collection = client.get_collection(config.collection_name)

# Get 200 chunks
sample = collection.get(limit=200, include=["metadatas"])
df = pd.DataFrame(sample['metadatas'])

print(f"📊 Analyzing {len(df)} chunks:")

# Check title issues
title_counts = df['episode_title'].value_counts(dropna=False)
empty_titles = (df['episode_title'] == '').sum()
null_titles = df['episode_title'].isna().sum()
valid_titles = len(df) - empty_titles - null_titles

print(f"Valid titles: {valid_titles}/{len(df)} ({valid_titles/len(df)*100:.1f}%)")
print(f"Empty titles (''): {empty_titles}/{len(df)} ({empty_titles/len(df)*100:.1f}%)")
print(f"Null titles: {null_titles}/{len(df)} ({null_titles/len(df)*100:.1f}%)")

# Show a few examples of each type
print(f"\n🔍 Examples:")
if valid_titles > 0:
    print("Valid title example:", df[df['episode_title'].notna() & (df['episode_title'] != '')]['episode_title'].iloc[0])
if empty_titles > 0:
    print("Empty title example:", df[df['episode_title'] == ''].iloc[0])
if null_titles > 0:
    print("Null title example:", df[df['episode_title'].isna()].iloc[0])

📊 Analyzing 200 chunks:
Valid titles: 200/200 (100.0%)
Empty titles (''): 0/200 (0.0%)
Null titles: 0/200 (0.0%)

🔍 Examples:
Valid title example: Banks, Bikies and Broadband


In [7]:
# Check what metadata fields actually exist vs what the code expects
sample = collection.get(limit=5, include=["metadatas"])
df = pd.DataFrame(sample['metadatas'])

print("🔍 Metadata fields in your chunks:")
print(df.columns.tolist())

print(f"\n🔍 Sample metadata:")
print(df.iloc[0].to_dict())

🔍 Metadata fields in your chunks:
['question_id', 'episode_title', 'speaker_type', 'speaker_name', 'response_id', 'episode_id', 'episode_date', 'speaker_id']

🔍 Sample metadata:
{'question_id': '4974.0', 'episode_title': 'Banks, Bikies and Broadband', 'speaker_type': 2, 'speaker_name': 'DIANE DENT', 'response_id': '153980', 'episode_id': '615', 'episode_date': '2009-04-09', 'speaker_id': ''}


In [8]:
#!/usr/bin/env python3
"""
🔍 Vector DB Diagnostic - Single Script Analysis
"""

from chromadb import PersistentClient
import os

# CONFIG
CHROMA_PATH = "data/chroma_db_mid_june9"
COLLECTION_NAME = "qa_optimized"

def diagnose_vector_db():
    print("🔍 Vector DB Diagnostic Report")
    print("=" * 50)
    
    # Connect and get basic stats
    client = PersistentClient(path=CHROMA_PATH)
    collection = client.get_collection(COLLECTION_NAME)
    count = collection.count()
    
    # Sample documents
    sample = collection.get(limit=3, include=["documents", "metadatas"])
    
    # Calculate sizes
    db_file = f"{CHROMA_PATH}/chroma.sqlite3"
    actual_size = os.path.getsize(db_file) / (1024 * 1024) if os.path.exists(db_file) else 0
    expected_size = count * 384 * 4 / (1024 * 1024)  # bge-small expected
    
    print(f"📊 Size Analysis:")
    print(f"  Documents: {count:,}")
    print(f"  Actual DB: {actual_size:.1f}MB")
    print(f"  Expected: {expected_size:.1f}MB")
    print(f"  Ratio: {actual_size/expected_size:.1f}x larger")
    
    print(f"\n🔍 Metadata Fields Found:")
    if sample['metadatas']:
        fields = list(sample['metadatas'][0].keys())
        print(f"  {fields}")
        
        # Check for specific field issues
        has_title = 'title' in fields
        has_episode_title = 'episode_title' in fields
        print(f"\n❌ Field Issues:")
        print(f"  Missing 'title': {not has_title}")
        print(f"  Has 'episode_title': {has_episode_title}")
        
    print(f"\n📝 Sample Document Length:")
    if sample['documents']:
        doc_len = len(sample['documents'][0])
        print(f"  First doc: {doc_len} chars")
        
    print(f"\n💡 Recommendation:")
    if actual_size > expected_size * 2:
        print("  🔄 REGENERATE - DB oversized")
    elif not has_title and has_episode_title:
        print("  🔧 QUICK FIX - Field rename needed")
    else:
        print("  ✅ LOOKS OK - Check pipeline code")

if __name__ == "__main__":
    diagnose_vector_db()

🔍 Vector DB Diagnostic Report
📊 Size Analysis:
  Documents: 99,439
  Actual DB: 370.9MB
  Expected: 145.7MB
  Ratio: 2.5x larger

🔍 Metadata Fields Found:
  ['episode_title', 'speaker_name', 'speaker_type', 'response_id', 'speaker_id', 'question_id', 'episode_date', 'episode_id']

❌ Field Issues:
  Missing 'title': True
  Has 'episode_title': True

📝 Sample Document Length:
  First doc: 151 chars

💡 Recommendation:
  🔄 REGENERATE - DB oversized


In [9]:
#!/usr/bin/env python3
"""
📊 Chunking Analysis - Current vs Optimized
"""

import duckdb
import pandas as pd

# Load your current filtered data
con = duckdb.connect("data/qanda_mid_june9.duckdb")
df = con.execute("""
    SELECT id, speaker_name, speaker_type, content, LENGTH(content) as content_length
    FROM fact_responses 
    WHERE LENGTH(content) >= CASE 
        WHEN speaker_type = 1 THEN 40
        WHEN speaker_type = 2 THEN 20  
        WHEN speaker_type = 3 THEN 12
        ELSE 20 END
""").df()

print("📊 Content Analysis:")
print(f"Documents after filtering: {len(df):,}")
print(f"Average content length: {df['content_length'].mean():.0f} chars")
print(f"Median content length: {df['content_length'].median():.0f} chars")
print(f"Max content length: {df['content_length'].max():,} chars")

# Analyze chunking behavior
def count_chunks_at_size(text, chunk_words):
    words = text.split()
    return max(1, len(words) // chunk_words + (1 if len(words) % chunk_words else 0))

# Test different chunk sizes
chunk_sizes = [200, 250, 300, 400]
print(f"\n📏 Chunks per Document (by size):")

for size in chunk_sizes:
    df[f'chunks_{size}'] = df['content'].apply(lambda x: count_chunks_at_size(x, size))
    avg_chunks = df[f'chunks_{size}'].mean()
    total_chunks = df[f'chunks_{size}'].sum()
    print(f"  {size} words: {avg_chunks:.1f} avg, {total_chunks:,} total")

# Analyze by speaker type
print(f"\n👥 By Speaker Type:")
for speaker_type in [1, 2, 3]:
    subset = df[df['speaker_type'] == speaker_type]
    avg_len = subset['content_length'].mean()
    avg_chunks_300 = subset['chunks_300'].mean()
    print(f"  Type {speaker_type}: {avg_len:.0f} chars avg, {avg_chunks_300:.1f} chunks")

print(f"\n💡 Optimization Recommendations:")
# Calculate prefix overhead
prefix_chars = 80  # Your current verbose prefix
total_docs = len(df)

for size in chunk_sizes:
    total_chunks = df[f'chunks_{size}'].sum()
    prefix_overhead_mb = (total_chunks * prefix_chars) / (1024 * 1024)
    print(f"  {size} words: {total_chunks:,} chunks, {prefix_overhead_mb:.1f}MB prefix overhead")

📊 Content Analysis:
Documents after filtering: 99,168
Average content length: 281 chars
Median content length: 128 chars
Max content length: 4,777 chars

📏 Chunks per Document (by size):
  200 words: 1.0 avg, 103,795 total
  250 words: 1.0 avg, 101,345 total
  300 words: 1.0 avg, 100,196 total
  400 words: 1.0 avg, 99,383 total

👥 By Speaker Type:
  Type 1: 152 chars avg, 1.0 chunks
  Type 2: 344 chars avg, 1.0 chunks
  Type 3: 338 chars avg, 1.0 chunks

💡 Optimization Recommendations:
  200 words: 103,795 chunks, 7.9MB prefix overhead
  250 words: 101,345 chunks, 7.7MB prefix overhead
  300 words: 100,196 chunks, 7.6MB prefix overhead
  400 words: 99,383 chunks, 7.6MB prefix overhead
